# Laboratorio: Diseñando Combos para "Café COVA" ☕

**Contexto:**
La gerencia de la cafetería de la facultad quiere aumentar sus ingresos. Saben que los estudiantes compran cosas en grupo, pero no saben exactamente qué. Tu trabajo como Ingeniero de Datos es leer los registros crudos de la caja registradora, transformar los datos a una matriz transaccional y aplicar el algoritmo Apriori para encontrar los "Combos" más rentables.

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# FASE 1: EXTRACCIÓN (Datos de producción)
# El sistema punto de venta (POS) exporta los tickets como una lista de listas.
tickets_cafeteria = [
    ['Cafe Americano', 'Croissant', 'Agua'],
    ['Frappe Mocha', 'Muffin', 'Agua'],
    ['Cafe Americano', 'Croissant', 'Jugo de Naranja'],
    ['Te Verde', 'Galletas', 'Agua'],
    ['Frappe Mocha', 'Muffin', 'Galletas'],
    ['Cafe Americano', 'Croissant', 'Agua', 'Galletas'],
    ['Frappe Mocha', 'Brownie', 'Agua'],
    ['Te Verde', 'Ensalada', 'Agua'],
    ['Cafe Americano', 'Brownie', 'Jugo de Naranja'],
    ['Frappe Mocha', 'Muffin', 'Brownie']
]

### Reto 1: Ingeniería de Características (De Lista a Matriz)
El algoritmo Apriori no entiende listas de texto. Necesitas convertir `tickets_cafeteria` en un DataFrame de Pandas donde cada columna sea un producto y cada fila (ticket) tenga un `True` o `False` (One-Hot Encoding transaccional).

In [ ]:
te = TransactionEncoder()
matriz_bool = te.fit_transform(tickets_cafeteria)
df_cafe_bool = pd.DataFrame(matriz_bool, columns=te.columns_)

display(df_cafe_bool)

### Reto 2: Extracción de Patrones Frecuentes
La gerencia te pide descartar las coincidencias raras. Extrae únicamente los `itemsets` (combinaciones de productos) que aparezcan en **al menos el 25% (0.25)** de los tickets.

In [ ]:
itemsets_cafe = apriori(df_cafe_bool, min_support=0.25, use_colnames=True)

display(itemsets_cafe)

### Reto 3: Creación de Reglas de Asociación
A partir de los itemsets frecuentes, genera las reglas de asociación. Filtra el resultado para mostrar **únicamente** aquellas reglas que tengan una `Confianza` superior al 60% (0.6).

In [ ]:
reglas_cafe = association_rules(itemsets_cafe, metric="confidence", min_threshold=0.6)

# Solo reglas con confianza mayor a 0.6
reglas_cafe = reglas_cafe[reglas_cafe['confidence'] > 0.6]
display(reglas_cafe[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

### 🧠 Reto 4: Defensa de Ingeniería (Análisis de Negocio)

Revisa la tabla resultante de tus reglas y responde las siguientes preguntas para el gerente de la cafetería:

1. **La regla de mayor Confianza:** Identifica la regla que tiene el mayor nivel de Confianza. Traduce esa regla al español simple (ej. "Si un alumno compra X, existe un Y% de probabilidad de que también compre Z").
2. **El engaño del Soporte vs Lift:** ¿Qué producto de la cafetería se vende muchísimo (alto soporte), pero no deberíamos ponerlo en promoción (combo) con el Frappe Mocha porque su Lift es bajo o cercano a 1? Explica matemáticamente por qué.

### Respuestas

**1. Regla de mayor confianza**

Hay un empate con confianza de 1.0 (100%):
- Croissant → Cafe Americano
- Muffin → Frappe Mocha

En español simple: *"Si un alumno compra un Muffin, existe un 100% de probabilidad de que también compre un Frappe Mocha"* (y lo mismo con el Croissant y el Cafe Americano).

In [ ]:
# Soporte, confianza y lift de Frappe Mocha -> Agua
soporte_agua = df_cafe_bool['Agua'].mean()
soporte_frappe = df_cafe_bool['Frappe Mocha'].mean()
soporte_ambos = (df_cafe_bool['Frappe Mocha'] & df_cafe_bool['Agua']).mean()

confianza = soporte_ambos / soporte_frappe
lift = confianza / soporte_agua

print(f"Soporte Agua: {soporte_agua}")
print(f"Confianza (Frappe Mocha -> Agua): {soporte_ambos} / {soporte_frappe} = {confianza:.2f}")
print(f"Lift (Frappe Mocha -> Agua): {confianza:.2f} / {soporte_agua} = {lift:.2f}")

**2. Soporte vs Lift**

El producto es el **Agua**. Es lo que más se vende (aparece en 6 de 10 tickets, soporte = 0.6), pero no conviene hacer combo con el Frappe Mocha.

- Confianza (Frappe → Agua) = 0.2 / 0.4 = 0.5
- Lift = 0.5 / 0.6 ≈ 0.83

El lift es menor a 1, o sea que comprar un Frappe no hace que la gente compre más Agua. El Agua se vende sola con casi todo, así que cuando aparece junto al Frappe es pura coincidencia. Por eso ni siquiera salió en las reglas. En cambio, Frappe Mocha → Muffin tiene lift de 2.5, ese sí es buen combo.